In [1]:
import torch
import myneural as mn
import torch.nn.functional as F
import time
import pyarrow.parquet as pq
import sentencepiece as spm
torch.manual_seed(42)



pf = pq.ParquetFile('data/tinystories0.parquet')


In [16]:
torch.__version__

'2.5.1'

In [2]:
with open('data/input.txt', 'w', encoding='utf-8') as f:
    for i in range(10):
        text_list = pf.read_row_group(i).column('text').to_pylist() # 1000 rows each row group     
        for story in text_list:
            f.write(story + "<|E|>\n")
# currently written: groups 1-10

In [3]:
spm.SentencePieceTrainer.train( # pyright: ignore[reportAttributeAccessIssue]
    input = 'data/input.txt',
    model_prefix = 'models/token_model',
    vocab_size = 4096,
    model_type = 'bpe',
    user_defined_symbols = ['<|E|>'],
    character_coverage = 1.0,
    byte_fallback = True,
    add_dummy_prefix = True
    )

In [4]:
sp = spm.SentencePieceProcessor(model_file="models/token_model.model") # pyright: ignore[reportCallIssue]
sp.Encode("Lily found a needle.<|E|>") # pyright: ignore[reportAttributeAccessIssue]

[359, 585, 262, 3757, 4023, 3]

In [5]:
text = open('data/input.txt', 'r').read()

In [6]:
def encode(text):
    return sp.Encode(text)

def decode(list):
    return sp.Decode(list)

In [7]:
encode("he saw something shiny! It was a golden penny!")

[281, 416, 552, 975, 4039, 422, 284, 262, 2745, 300, 2667, 4039]

In [8]:
config = mn.Config()

In [9]:

config.n_embd = 256
config.context_length = 256
config.n_head = 8
config.n_layer = 8
config.batch_size = 64
config.device = 'cuda'
max_iter = 6000
config.vocab_size = 4096

In [10]:
data = torch.tensor(encode(text), dtype=torch.long, device=config.device)

n = int(0.9 * len(data))
train = data[:n]
val = data[n:]

def get_batch(split):
    x = train if split == 'train' else val
    ix = torch.randint(0, len(x) - config.context_length, (config.batch_size,), device=config.device)
    xs = torch.stack([x[i:i+config.context_length] for i in ix]) # (B, T)
    ys = torch.stack([x[i+1:i+config.context_length+1] for i in ix]) # (B, T)
    return xs, ys

In [11]:
model = mn.Transformer(config, dtype=torch.float32, device=config.device)


In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4)

In [13]:
model = torch.compile(model)

In [14]:
# # Load the list of tensors
# loaded_params = torch.load("models/params_new.pth")

# # Manually re-assign them to your model
# for p, loaded_p in zip(model.parameters(), loaded_params):
#     p.data.copy_(loaded_p.data)

In [15]:
for i in range(max_iter):
    t_start = time.time()
    # forward pass
    xs, ys = get_batch('train')
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits, loss = model(xs, ys)
    
    # backward pass
    for p in model.parameters():
        p.grad = None
    loss.backward()
    
    # update
    optimizer.step()
    if i % 25 == 0:
        torch.cuda.synchronize()
        t_end = time.time()
        print(f"Step {i}: Loss = {loss.item()} | {(t_end - t_start)*1000.0:.3f}ms")

BackendCompilerFailed: backend='inductor' raised:
RuntimeError: Cannot find a working triton installation. Either the package is not installed or it is too old. More information on installing Triton can be found at https://github.com/openai/triton

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True


In [ ]:
params_list = list(model.parameters())
torch.save(params_list, "models/params_new.pth")

In [ ]:
xs, ys = get_batch('val')

logits, loss = model.forward(xs, ys)
loss

OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 14.20 GiB is allocated by PyTorch, and 123.68 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
print(decode(model.generate(torch.tensor([[3]], dtype=torch.long, device=config.device), 2000)[0].tolist()))

<|E|> One day, a boy named Terry and his mom had a great idea. They decided to go to the restaurant maker. Terry was ordinary. He was excited to go shopping through the computers and pay to his destination. When they arrived, Terry was so excited! He quickly grabbed a handful of them and took one. He watched as the orderer crer fixed his ear and flew off to the top with the pictures. It was the perfect way to stay. Everyone was so happy as the food was sent to that paste! They danced and played together all day. When it was time to go home, Terry didn't want to leave. Suddenly, he heard an urge of farm. He hopped away to find another place in the big field! Terry was so happy, he jumped up and gave the people a big welcome. Everyone was so surprised to see the sandwiches and the other chickens home in its mind. Everyone was very happy, and they were never envious of the new toys!<|E|> Bob went to visit the chubby creeker. He was very excited. The creeker showed him lots of interesting 